In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import re
import random
import unicodedata

# ============================================================
# 0. PBP 파일명 패턴
# ============================================================
PBP_PATTERN = re.compile(r'^\d{4}_\d{2}_[a-z]+\.xlsx$')


# ============================================================
# 1. game_idx 수집 (기존 PBP 파일 기반)
# ============================================================
def get_game_idx_from_pbp_folders(base_dir, years=None):
    if years is None:
        years = range(2022, 2023)

    game_idx_by_year = {}

    for year in years:
        year_dir = os.path.join(base_dir, str(year))
        if not os.path.exists(year_dir):
            print(f"{year}년 폴더 없음, 스킵")
            continue

        ids = []

        for root, dirs, files in os.walk(year_dir):
            dirs[:] = [unicodedata.normalize('NFC', d) for d in dirs]

            for fname in files:
                fname = unicodedata.normalize('NFC', fname)

                if fname.startswith('~$'):
                    continue
                if not PBP_PATTERN.match(fname):
                    continue

                fpath = os.path.join(root, fname)
                try:
                    df = pd.read_excel(fpath, usecols=['game_id'])
                    game_ids = df['game_id'].dropna().astype(int).unique().tolist()
                    ids.extend(game_ids)
                except Exception as e:
                    print(f"  [{fname}] 읽기 실패: {e}")
                    continue

        ids = sorted(set(ids))
        game_idx_by_year[year] = ids
        print(f"{year}년: {len(ids)}개 game_idx 수집")

    return game_idx_by_year


# ============================================================
# 2. 박스스코어 페이지 파싱 (이닝별 점수)
# ============================================================
def parse_inning_summary(game_idx, session=None):
    url = f"https://www.korea-baseball.com/game/box_score?game_idx={game_idx}"

    try:
        if session:
            res = session.get(url, timeout=10)
        else:
            res = requests.get(url, timeout=10)
        res.raise_for_status()
    except Exception as e:
        print(f"  [{game_idx}] 요청 실패: {e}")
        return []

    soup = BeautifulSoup(res.text, 'html.parser')

    # ── 팀명 파싱
    hero = soup.select('.hero_sum .team')
    team1 = hero[0].select_one('.txt').get_text(strip=True) if len(hero) > 0 else None
    team2 = hero[1].select_one('.txt').get_text(strip=True) if len(hero) > 1 else None

    if not team1 or not team2:
        print(f"  [{game_idx}] 팀명 파싱 실패")
        return []

    # ── 이닝 테이블: tb_head(신) → box_table(구) 순으로 시도
    table = soup.select_one('.tb_head table')
    if not table:
        table = soup.select_one('.box_table table')
    if not table:
        print(f"  [{game_idx}] 이닝 테이블 없음")
        return []

    rows = table.select('tbody tr')
    if len(rows) < 2:
        print(f"  [{game_idx}] tbody 행 부족 ({len(rows)}행)")
        return []

    records = []

    for row_idx, row in enumerate(rows[:2]):  # top/bot 두 행만
        topbot = 'top' if row_idx == 0 else 'bot'
        team   = team1 if row_idx == 0 else team2

        cells = row.find_all(['th', 'td'])
        if len(cells) < 6:
            continue

        inning_cells = cells[1:-4]   # 이닝 셀 (최대 20개)
        rheb_cells   = cells[-4:]    # R, H, E, B

        def parse_int(cell):
            txt = re.sub(r'[^\d]', '', cell.get_text(strip=True))
            return int(txt) if txt else 0

        R = parse_int(rheb_cells[0])
        H = parse_int(rheb_cells[1])
        E = parse_int(rheb_cells[2])
        B = parse_int(rheb_cells[3])

        # 이닝별 점수: 빈 셀은 0으로, 연속 2개 빈 셀이면 중단
        empty_count = 0
        for inning_num, cell in enumerate(inning_cells, start=1):
            txt = cell.get_text(strip=True)
            if txt == '':
                empty_count += 1
                if empty_count >= 2:
                    break
                continue
            else:
                empty_count = 0

            score = int(txt) if txt.isdigit() else 0

            records.append({
                'game_id': game_idx,
                'team1':   team1,
                'team2':   team2,
                'team':    team,
                'topbot':  topbot,
                'inning':  inning_num,
                'score':   score,
                'R':       R,
                'H':       H,
                'E':       E,
                'B':       B,
            })

    return records


# ============================================================
# 3. 저장 / 기존 파일 로드
# ============================================================
def _save_inning_summary(records, output_path):
    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)

    col_order = ['game_id', 'team1', 'team2', 'team', 'topbot',
                 'inning', 'score', 'R', 'H', 'E', 'B']
    df = df[[c for c in col_order if c in df.columns]]

    df.to_excel(output_path, index=False)
    return df


def _load_existing_records(output_dir, year):
    """기존 inning_summary_{year}.xlsx 가 있으면 레코드 + 이미 수집된 game_id 셋 반환"""
    output_path = os.path.join(output_dir, f'inning_summary_{year}.xlsx')

    if not os.path.exists(output_path):
        return [], set()

    try:
        df = pd.read_excel(output_path)
        existing_ids = set(df['game_id'].dropna().astype(int).unique().tolist())
        records = df.to_dict('records')
        return records, existing_ids
    except Exception as e:
        print(f"  [{year}] 기존 파일 읽기 실패: {e}")
        return [], set()


# ============================================================
# 4. 메인 크롤링
# ============================================================
def crawl_inning_summary_by_year(game_idx_by_year, output_dir,
                                 save_every=50, max_retry=3):
    os.makedirs(output_dir, exist_ok=True)

    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    })

    total_failed = {}

    for year, game_idx_list in sorted(game_idx_by_year.items()):
        output_path = os.path.join(output_dir, f'inning_summary_{year}.xlsx')

        # 기존 파일에서 이미 수집된 game_id는 스킵 대상으로 분리
        all_records, existing_ids = _load_existing_records(output_dir, year)
        target_list = [gid for gid in game_idx_list if gid not in existing_ids]

        print(f"\n{'='*50}")
        print(f"{year}년: 전체 {len(game_idx_list)}경기 / "
              f"기존 {len(existing_ids)}개 스킵 / 신규 {len(target_list)}개 수집")
        print(f"{'='*50}")

        failed = []

        for i, game_idx in enumerate(target_list):

            records = []
            elapsed = 0
            for attempt in range(1, max_retry + 1):
                start = time.time()
                records = parse_inning_summary(game_idx, session)
                elapsed = time.time() - start

                if records:
                    print(f"  [{i+1}/{len(target_list)}] {game_idx} → {len(records)}행 ({elapsed:.1f}s)")
                    break
                else:
                    if attempt < max_retry:
                        retry_wait = random.uniform(9.0, 11.0)
                        print(f"  [{i+1}/{len(target_list)}] {game_idx} → 실패 ({elapsed:.1f}s) "
                              f"/ 재시도 {attempt}/{max_retry} ({retry_wait:.1f}s 후)")
                        time.sleep(retry_wait)
                    else:
                        print(f"  [{i+1}/{len(target_list)}] {game_idx} → 최종 실패 ({max_retry}회 시도)")
                        failed.append(game_idx)

            if records:
                all_records.extend(records)

            # 중간 저장 (기존 레코드 + 신규 레코드 누적본을 그대로 저장)
            if (i + 1) % save_every == 0:
                _save_inning_summary(all_records, output_path)
                print(f"  >>> 중간 저장 완료 ({i+1}/{len(target_list)})")

            # 9~11초 랜덤 딜레이 (응답시간 포함)
            target = random.uniform(9.0, 11.0)
            wait = max(0, target - elapsed)
            time.sleep(wait)

        # 최종 저장
        df = _save_inning_summary(all_records, output_path)
        total_failed[year] = failed

        print(f"\n{year}년 완료: {len(df)}행 저장 → {output_path}")
        print(f"신규 수집 실패 {len(failed)}개: {failed if failed else '없음'}")

    return total_failed


# ============================================================
# 실행
# ============================================================
BASE_DIR = '/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)'
OUT_DIR  = os.path.join(BASE_DIR, 'inning_summary')

# game_idx 수집
# 전체: years=None  /  특정: years=[2022]  /  복수: years=[2022, 2023]
game_idx_by_year = get_game_idx_from_pbp_folders(BASE_DIR, years=[2022])

# 수집 현황 확인
print("\n[수집 현황]")
for year, ids in sorted(game_idx_by_year.items()):
    print(f"  {year}년: {len(ids)}경기")

# 크롤링 시작 (이미 inning_summary 파일에 있는 game_id는 자동 스킵)
failed = crawl_inning_summary_by_year(
    game_idx_by_year,
    output_dir=OUT_DIR,
    save_every=50,
    max_retry=3,
)

# 최종 실패 목록
print("\n[최종 실패 목록]")
for year, ids in failed.items():
    if ids:
        print(f"  {year}년: {ids}")